# Tutorial 1 — Muon-neutrino disappearance

This is the original working tutorial for [Machine Learning Neutrino-Nucleus Cross Sections](https://arxiv.org/abs/2412.16303). It learns a cross section from a near-detector (ND) lepton distribution, applies it to a far-detector (FD) flux, and compares oscillation fits using the known and learned cross sections.

Install from the repository root with `python -m pip install -e '.[notebook]'`, open this notebook there, and choose **Restart Kernel and Run All**. All input data are included. No saved model or FD samples are required.

`PROFILE="quick"` reduces the numerical workload. `"full"` increases the statistical and numerical budgets. Quick plots demonstrate the workflow; they are not the paper's confidence intervals. The default five-output parametrization spans the single-lepton model class; `N_SF=3` selects the minimal basis. The current library uses the later toy-model prescription and VarPro training, so this is a closure-test tutorial rather than an exact reproduction of the published Fig. 6. See [the audit](docs/tutorial1-reproduction.md).


## Setup: local CPU or Google Colab

Locally, install with `python -m pip install -e '.[notebook]'` from the cloned repository, then open this notebook in Jupyter. The setup cell finds the repository root automatically.

In Colab, choose a Python runtime with **no accelerator** and run this setup cell first. It clones `main`, installs the package, and makes its input files available. If this repository is private, select `COLAB_SOURCE="upload"` and upload its repository ZIP when prompted. Instructions are in the [README](https://github.com/karlaTame/ML_nuxsec_school/blob/main/README.md). Start with `PROFILE="smoke"` in the next configuration cell for a short execution check; use `"quick"` for the tutorial. CPU is sufficient for both.

Runtime outputs stay under `outputs/`; source notebooks ship without outputs or saved machine-specific kernels. Colab runtime files are temporary, so download any results you want to retain before disconnecting.


In [ ]:
# Local: install .[notebook] first. Colab: this cell prepares the checkout.
from pathlib import Path
import importlib
import os
import shutil
import subprocess
import sys
import tempfile
import zipfile

REPO_URL = "https://github.com/karlaTame/ML_nuxsec_school.git"
REPO_BRANCH = "main"  # change this when using a fork or another branch
COLAB_SOURCE = "github"  # github | upload (ZIP of this repository, for private repos)


def prepare_colab(repo_url, branch, target, source="github", archive=None):
    target = Path(target).resolve()
    if not (target / "pyproject.toml").is_file():
        if target.exists():
            raise RuntimeError(f"{target} exists but is not this repository; choose another folder.")
        if source == "upload":
            if archive is None:
                from google.colab import files
                uploaded = files.upload()  # choose the repository ZIP, not just the notebook
                if len(uploaded) != 1:
                    raise ValueError("Upload exactly one ZIP of the repository.")
                archive = next(iter(uploaded))
            with tempfile.TemporaryDirectory() as staging:
                staging = Path(staging)
                with zipfile.ZipFile(archive) as bundle:
                    for member in bundle.infolist():
                        if not (staging / member.filename).resolve().is_relative_to(staging.resolve()):
                            raise ValueError("ZIP contains a path outside its extraction folder.")
                    bundle.extractall(staging)
                roots = [p.parent for p in staging.rglob("pyproject.toml")
                         if (p.parent / "src" / "ml4nu").is_dir()]
                if len(roots) != 1:
                    raise ValueError("ZIP must contain exactly one complete tutorial repository.")
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(str(roots[0]), str(target))
        elif source == "github":
            try:
                subprocess.run(["git", "clone", "--depth", "1", "--branch", branch,
                                repo_url, str(target)], check=True)
            except subprocess.CalledProcessError as error:
                raise RuntimeError("Clone failed. For a private repository, set COLAB_SOURCE='upload' "
                                   "and upload the ZIP of this branch. Check repository access and branch name.") from error
        else:
            raise ValueError("COLAB_SOURCE must be 'github' or 'upload'.")
    if not (target / "src" / "ml4nu").is_dir():
        raise RuntimeError("The selected folder is not a complete tutorial checkout.")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e",
                    str(target) + "[colab]"], check=True)
    # pip's editable .pth is processed only when Python starts; this kernel is already running.
    sys.path.insert(0, str(target / "src"))
    importlib.invalidate_caches()
    os.chdir(target)
    return target


try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    ROOT = prepare_colab(REPO_URL, REPO_BRANCH, "/content/ML_nuxsec_school", COLAB_SOURCE)
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "pyproject.toml").is_file() and (p / "src" / "ml4nu").is_dir()), None)
    if ROOT is None:
        raise RuntimeError("Open this notebook from your cloned ML_nuxsec_school repository.")
    os.chdir(ROOT)
print("Repository:", ROOT)


In [ ]:
from pathlib import Path
import os
import torch

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError("Start Jupyter in the ml4nu repository root.")
PROFILE = os.environ.get("ML4NU_PROFILE", "quick")  # quick | full | smoke
SETTINGS = {
    "smoke": dict(steps=40, events=200, boots=32, enu_fd=48, v_fd=24, osc=12),
    "quick": dict(steps=5000, events=1000, boots=128, enu_fd=128, v_fd=64, osc=40),
    "full": dict(steps=100000, events=6200, boots=25000, enu_fd=512, v_fd=256, osc=100),
}
if PROFILE not in SETTINGS:
    raise ValueError(f"Unknown profile: {PROFILE}")
cfg = dict(SETTINGS[PROFILE])
# Optional controlled FD-statistics comparison without retraining.
for key, name in (("steps", "ML4NU_TRAIN_STEPS"), ("events", "ML4NU_FD_EVENTS"), ("boots", "ML4NU_N_BOOTS")):
    if name in os.environ:
        cfg[key] = int(os.environ[name])
CHECKPOINT = os.environ.get("ML4NU_CHECKPOINT")
OUT = ROOT / "outputs" / "tutorial1" / PROFILE
if "ML4NU_FD_EVENTS" in os.environ or "ML4NU_N_BOOTS" in os.environ:
    OUT = OUT.with_name(f"{PROFILE}_N{cfg['events']}_B{cfg['boots']}")
OUT.mkdir(parents=True, exist_ok=True)
torch.set_default_device("cpu")
torch.set_num_threads(2)
print(PROFILE, cfg)


In [ ]:
from ml4nu.pdg import pid, masses
from ml4nu.flux import FluxNearDetector, FluxFarDetector
import numpy as np
import matplotlib.pyplot as plt
from ml4nu.structure_functions import PDFInterpolator, StructureFunctions
from ml4nu.kinematics import KinematicMapping, KinematicsGrid, make_newk2, unmake_newk2, make_kinematic_factors_newk2, jacobian_newk2
from ml4nu.cross_section import TruthCalc, ModelCalc
from ml4nu.utils import make_marginal_rate, make_mlp, grab, safe_make_dx, precompute_bootstrap_tensor, make_symm_bin_edges
from ml4nu.sampling import accept_reject, fmax, Sampler
from ml4nu.far_detector_fit import make_I, make_dsigma_samp, find_min_2d, make_log_likelihood, get_grid_min_coords
from ml4nu.nufast import Probability_Matter_LBL

import copy
from tqdm import tqdm
import scipy

import torch
torch.set_default_dtype(torch.float64)
torch.set_grad_enabled(False)
from ml4nu.single_channel import (in_lepton_region, shape_log_likelihood, likelihood_grid,
    bootstrap_minima, boundary_counts, bootstrap_plot_limits, bootstrap_kde)
import json

# Sampling also stays on CPU when Colab happens to expose a GPU.
from ml4nu import sampling
sampling.device = torch.device("cpu")


In [ ]:
plot_rc_params = {
#     'mathtext.fontset' : 'stix',
    'font.family' : 'STIXGeneral',
    'mathtext.fontset' : 'cm',
#     'axes.formatter.use_mathtext' : True
}

## Flux Exploration

Compare the unoscillated and oscillated fluxes. This original single-channel example uses the ND flux shape before oscillating it; Tutorial 2 uses separate ND and FD tables.


In [ ]:
global_datadir = "./data/"
flux_file = (global_datadir + "histos_g4lbne_v3r5p4_QGSP_BERT_OptimizedEngineeredNov2017_neutrino_LBNEND_globes_flux.txt")

# Create a FluxNearDetector object
nd_flux = FluxNearDetector(flux_file, force_emax=10)

# Create a FluxFarDetector object
PAPER1_OSC = dict(s12=0.554076, s23=0.748999, s13=0.148155,
                 dCP=3.08923, d21=7.49e-5, d31=2.534e-3)
# Paper 1 keeps only nu_mu -> nu_mu. Use the same flux in sampling and fitting.
fd_flux = FluxFarDetector(nd_flux=nd_flux, osc_params=PAPER1_OSC,
                         osc_flavors=[pid.numu])

# Backward-horn (antineutrino-mode) ND flux — used when TRAIN_BWD is on
flux_file_bwd = (global_datadir + "histos_g4lbne_v3r5p4_QGSP_BERT_OptimizedEngineeredNov2017_antineutrino_LBNEND_globes_flux.txt")
nd_flux_bwd = FluxNearDetector(flux_file_bwd)

In [ ]:
energies = np.linspace(0,10,10000)
plt.plot(energies, nd_flux(pid.numu, energies), label="ND mu")
plt.plot(energies, fd_flux(pid.numu, energies), label="FD mu")
plt.plot(energies, nd_flux(pid.nue, energies), label="ND e")
plt.plot(energies, fd_flux(pid.nue, energies), label="FD e")
plt.plot(energies, fd_flux(pid.nutau, energies), label="FD tau")
plt.legend()
plt.yscale("log")
plt.xlim(0,10)
plt.ylim(10**(-7.7), 10**(-3))
plt.title("Neutrino fluxes")
plt.xlabel(r"$E_{\nu}$", fontsize=14)
plt.ylabel(r"$\Phi(E_{\nu})$", fontsize=14);

In [ ]:
# plot antineutrino mode fluxes

energies = np.linspace(0,10,10000)
plt.plot(energies, nd_flux(pid.numubar, energies), label="ND mu")
plt.plot(energies, fd_flux(pid.numubar, energies), label="FD mu")
plt.plot(energies, nd_flux(pid.nuebar, energies), label="ND e")
plt.plot(energies, fd_flux(pid.nuebar, energies), label="FD e")
plt.plot(energies, fd_flux(pid.nutaubar, energies), label="FD tau")
plt.legend()
plt.yscale("log")
plt.xlim(0,10)
plt.ylim(10**(-7.7), 10**(-3))
plt.title("Anti-neutrino fluxes")
plt.xlabel(r"$E_{\nu}$", fontsize=14)
plt.ylabel(r"$\Phi(E_{\nu})$", fontsize=14);

## Cross Section

In [ ]:
# ND Settings

# Training grid
N_Enu_ND = 32
N_v_ND = 16
v1 = torch.linspace(0.25, 3.25, N_v_ND)
v2 = torch.linspace(0, 0.65, N_v_ND)
enu = torch.linspace(masses[pid.mu]+1e-6, 10, N_Enu_ND)

# Training settings
N_steps = cfg["steps"]

# ── Training-mode toggles ──
# N_SF: 3 = minimal single-lepton basis (classic paper-1 form);
#   5 = overparameterized basis — for a single lepton it spans the SAME
#   model class (the unique reduction is exact), but the redundancy acts as
#   an implicit regularizer: ~1.7x better FD-flux fidelity at equal steps.
# TRAIN_BWD: also fit the antineutrino-mode (RHC) numu marginal — a second
#   flux weighting that genuinely constrains the E-shape (another ~1.7x;
#   assumes wrong-sign muon ID at the ND, like the multi-channel study).
# Both off reproduces the classic single-channel setup and its known
# E-degeneracy (sigma(E) can go negative below ~2 GeV).
N_SF = 5
TRAIN_BWD = False
clip_norm = np.inf #torch.tensor(np.inf)

In [ ]:
# compute kinematic variables on (v1, v2) grid

kin_mapping = KinematicMapping(unmake_func=unmake_newk2, make_kinematic_factors=make_kinematic_factors_newk2,
                               jacobian=jacobian_newk2)
kin_grid = KinematicsGrid(v1, v2, enu, pid.mu, kin_mapping=kin_mapping)

## Toy Model

The supplied PDF grid defines synthetic truth. Its use here is a closure test: the target is known, so extrapolation can be checked directly.


In [ ]:
# toy model

pdf = PDFInterpolator('data/pdf/pdf_x1024_q1024.npy', clamp_q2=True, clamp_x=True)
sf = StructureFunctions(pdf)
# define function truth(Enu, v1, v2, flavor, kinematic_mapping) that returns true cross section
truth = TruthCalc(sf, enumax=10, device="cpu")

In [ ]:
# evaluate toy model flux-averaged cross section on kinematic grid

dsigma = truth(kin_grid.grid_enu, kin_grid.grid1, kin_grid.grid2, flavor=pid.mu, kin_mapping=kin_mapping)

phi_ND_train = nd_flux(pid.numu, kin_grid.all_enu)

# marginal distribution is target for training
P = make_marginal_rate(dsigma, phi_ND_train, kin_grid.denu, kin_grid.dV2)

# 3d cross-section (only accessible for toy model)
P3 = dsigma * phi_ND_train[None,None,:]

# Backward-horn flux + marginal target (cheap; used when TRAIN_BWD is on)
phi_ND_train_bwd = nd_flux_bwd(pid.numu, kin_grid.all_enu)
P_bwd = make_marginal_rate(dsigma, phi_ND_train_bwd, kin_grid.denu, kin_grid.dV2)

# Precompute quantities used in training
S = dsigma / (dsigma.sum() * kin_grid.dV3)

dEnu = kin_grid.denu.item()
dV2 = kin_grid.dV2.item()
dV3 = kin_grid.dV3.item()

In [ ]:
# Plot total muon cross section
S_mu = grab(dsigma.sum((0,1)) * kin_grid.dV2)
S_mu = S_mu/S_mu.max()
plt.plot(kin_grid.all_enu, S_mu/S_mu.sum()/dEnu)
plt.xlabel(r'$E_\nu$ [GeV]')
plt.ylabel(r'$\sigma(E_\nu) / \int dE_\nu \sigma(E_\nu)$')
plt.ylim(0,0.25)
plt.show()

In [ ]:
print(torch.min(dsigma))

## Model and Training

Known kinematic coefficients turn the network outputs (structure functions) into a cross section. Training sees only the flux-integrated ND distribution. Variable projection (VarPro) solves linear output coefficients at each step while Adam learns network features.


In [ ]:
# Model — unified SFModel from the library (ml4nu.model), with the VarPro
# linear-coefficient buffer c solved analytically each training step.
# N_SF (settings cell) picks the basis: 3-SF pairs with unique=True
# kinematic factors, 5-SF with unique=False — same single-lepton model
# class, different optimization behavior.
from ml4nu.model import make_sf_model
from ml4nu import varpro

# make_sf_model seeds torch.manual_seed(17) then constructs, reproducing the
# old cell's init (and its RNG stream) bit-exactly.
model = make_sf_model(n_sf=N_SF, seed=17)

In [ ]:
# Plotting scripts
def plot_2d_density(P, x, y, **kwargs):
    P, x, y = grab(P), grab(x), grab(y)
    assert len(P.shape) == 2
    assert len(x.shape) == len(y.shape) == 1
    assert P.shape == (len(x), len(y))
    return plt.imshow(P.T,
        extent=[x.min(), x.max(), y.min(), y.max()],
        aspect='auto', origin='lower',
        **kwargs
    )

def plot_2d_marginal_model_comparison(kin_grid, phi_ND, data_dist, dsigma_):
    Q = grab( make_marginal_rate(dsigma_.detach(), phi_ND, kin_grid.denu, kin_grid.dV2) )
    P = grab( data_dist )


    fig, axes = plt.subplots(1,3, figsize=(8,2.5), sharex=True, sharey=True)
    plt.sca(axes[0])
    plot_2d_density(P, kin_grid.all_var1, kin_grid.all_var2)
    plt.colorbar()
    plt.sca(axes[1])
    plot_2d_density(Q, kin_grid.all_var1, kin_grid.all_var2)
    plt.colorbar()
    plt.sca(axes[2])
    plot_2d_density(P-Q, kin_grid.all_var1, kin_grid.all_var2)
    plt.colorbar()
    d = grab(P-Q)
    print(d.mean(), d.std(), (d**2).mean(), (d**2).std())
    plt.tight_layout(h_pad=0.1, w_pad=0.1)

In [ ]:
# Plotting the cross section BEFORE training
model_calc = ModelCalc(model, enumax=10, unique=(N_SF == 3), device="cpu")
dsigma_ = model_calc(kin_grid.grid_enu, kin_grid.grid1, kin_grid.grid2, flavor=pid.mu, kin_mapping=kin_mapping)

In [ ]:
plot_2d_marginal_model_comparison(kin_grid, phi_ND_train, P, dsigma_)

In [ ]:
# VarPro + ReduceLROnPlateau training, backported from the MINERvA study
# (ml4nu.varpro.fit_varpro). Replaces plain Adam + the hand-rolled
# lr *= 2/3-every-10k schedule + disabled gradient clipping; the loss is the
# SAME marginal MSE (identity-covariance chi^2 = ((m - P)**2).sum(), pinned
# by tests/test_varpro_dune.py). c is solved analytically each step, so no
# manual norm initialisation is needed.
phi_ND_train = torch.as_tensor(phi_ND_train)

ds_nd = varpro.make_marginal_dataset('DUNE_ND_numu', model_calc, kin_grid,
                                     phi_ND_train, P, kin_mapping=kin_mapping)
train_datasets = [ds_nd]
if TRAIN_BWD:
    phi_ND_train_bwd = torch.as_tensor(phi_ND_train_bwd)
    train_datasets.append(varpro.make_marginal_dataset(
        'DUNE_ND_numu_bwd', model_calc, kin_grid,
        phi_ND_train_bwd, P_bwd, kin_mapping=kin_mapping))

def shape_metric(m):
    # truth-shape diagnostic (the old loop's test_loss), evaluated no-grad
    dsig = model_calc.dsigma()
    S_ = dsig / (dsig.sum() * dV3)
    return ((S_ - S) ** 2).sum().item() * dV3

# loss_norm='auto': the marginal-rate MSE is O(1e-11) and its ~1e-9
# gradients sit below Adam's eps=1e-8, which silently cripples training
# (the production trainer uses LOSS_NORM=1e11 for the same reason).
# 'auto' rescales so the objective starts at 1.
if CHECKPOINT:
    saved = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
    if saved["meta"]["n_sf"] != N_SF:
        raise ValueError("Checkpoint structure-function count differs from N_SF.")
    if saved["meta"]["settings"]["steps"] != cfg["steps"]:
        raise ValueError("Checkpoint training steps differ from this profile.")
    best_loss_model = saved["model"]
    training_history = saved["history"]
    best_loss_step = saved["meta"]["best_loss_step"]
    print("Reusing trained model:", CHECKPOINT, "(best step", best_loss_step, ")")
else:
    result = varpro.fit_varpro(model, train_datasets,
                               extra_metrics={'shape': shape_metric},
                               best_metric='shape', loss_norm='auto',
                               n_steps=N_steps, lr=1e-3, clip_norm=1.0,
                               log_every=500)
    best_loss_model = result.best_loss_model
    training_history = result.history
    best_loss_step = result.best_loss_step

loss_history = training_history['loss']
test_history = training_history['shape']
norm_history = training_history['grad_norm']


In [ ]:
# Plotting the Loss Function

fig, axes = plt.subplots(3,1, sharex=True, figsize=(5,6))
plt.sca(axes[0])
plt.plot(loss_history)
plt.semilogy()
plt.ylabel('Loss')
plt.sca(axes[1])
plt.plot(test_history)
plt.semilogy()
plt.ylabel('err dsigma')
plt.sca(axes[2])
plt.plot(norm_history)
plt.semilogy()
plt.ylabel('grad norm')
plt.tight_layout(h_pad=0.1)

In [ ]:
# Plotting the cross section AFTER training
model.load_state_dict(best_loss_model)
# Re-solve the linear coefficients for the selected weights.
varpro.resolve_c(model, train_datasets)
dsigma_ = model_calc(kin_grid.grid_enu, kin_grid.grid1, kin_grid.grid2, flavor=pid.mu, kin_mapping=kin_mapping)
plot_2d_marginal_model_comparison(kin_grid, phi_ND_train, P, dsigma_)
# Keep expensive ND training separate from FD / plotting experiments.
torch.save(dict(model={k: v.detach().cpu() for k, v in model.state_dict().items()},
                history=training_history,
                meta=dict(n_sf=N_SF, settings=cfg, best_loss_step=best_loss_step,
                          nd_grid=[N_Enu_ND, N_v_ND], seed=17)), OUT / "nd_checkpoint.pt")


In [ ]:
# correct overall sign of xsec if needed (it will be clear whether this is
# needed when plotting I below). The overall norm now lives in the VarPro
# coefficients c, which are solved against the (positive) ND marginal, so no
# flip should be needed; set SIGN_FLIP = -1 to force one.
SIGN_FLIP = 1
with torch.no_grad():
    model.c.mul_(SIGN_FLIP)

## Oscillation Analysis

Freeze the trained network, generate synthetic FD events, and fit oscillation parameters with both cross sections on the same data. This original analysis uses a normalized, shape-only likelihood. It does not fit the CP phase.


In [ ]:
# FD Settings

# statistics
N_samps = cfg["events"]

# resampling
N_boots = cfg["boots"]

# oscillation analysis grid
N_Enu_FD = cfg["enu_fd"]
N_v_FD = cfg["v_fd"]

In [ ]:
# Generate events only inside the same lepton region used by the FD normalizer.
nsamples = 2 * N_samps
nbatch = 16384
np.random.seed(11)
torch.manual_seed(11)
V1_ACCEPT = (float(kin_grid.all_var1.min()), float(kin_grid.all_var1.max()))
V2_ACCEPT = (float(kin_grid.all_var2.min()), float(kin_grid.all_var2.max()))

def truth_in_analysis_region(enu, emu, cost, *, flavor):
    rate = truth(enu, emu, cost, flavor=flavor)
    keep = in_lepton_region(torch.as_tensor(emu), torch.as_tensor(cost), V1_ACCEPT, V2_ACCEPT)
    return torch.where(keep.to(rate.device), rate, torch.zeros_like(rate))

sampler = Sampler(fd_flux, truth_in_analysis_region, enu_window=(float(kin_grid.all_enu.min()), 10.))
# The unrestricted density bounds the restricted one, so the envelope search
# is not disrupted by a discontinuity at the lepton-acceptance boundary.
unrestricted_sampler = Sampler(fd_flux, truth)
max_val = 1.1 * fmax(f=lambda x: unrestricted_sampler.proposal_density(x[0], x[1], x[2]),
                    ranges=[(masses[pid.mu], 10), (masses[pid.mu], 10), (-1, 1)])
samples_FD = accept_reject(lambda count: sampler.propose(count, flavor=pid.numu),
                           nsamples, max_val, nbatch)
print("FD sample lepton acceptance:", V1_ACCEPT, V2_ACCEPT)


In [ ]:
# Plot samples
fig, axes = plt.subplots(1,3,figsize=(15,6))
plt.sca(axes[0])
plt.hist(samples_FD[:, 0], bins=75, histtype='step')
plt.xlabel(r'$E_\nu$ [GeV]')
plt.sca(axes[1])
plt.hist(samples_FD[:, 1], bins=75, histtype='step')
plt.xlabel(r'$E_\mu$ [GeV]')
plt.sca(axes[2])
plt.hist(samples_FD[:, 2], bins=75, histtype='step')
plt.xlabel(r'$\cos\theta_\mu$')
plt.tight_layout(h_pad=0.1)

In [ ]:
# take desired subset of FD samples
trim = samples_FD
trim = trim[:N_samps]
print("Number of FD samples = ", len(trim))

# unpack
samp_Enu, samp_Emu, samp_cost = torch.tensor(trim.T)
samp_v1, samp_v2 = make_newk2(samp_Emu, samp_cost)

# Setup grid
all_Enu = torch.linspace(kin_grid.all_enu.min(), kin_grid.all_enu.max(), N_Enu_FD)
all_v1 = torch.linspace(kin_grid.all_var1.min(), kin_grid.all_var1.max(), N_v_FD)
all_v2 = torch.linspace(kin_grid.all_var2.min(), kin_grid.all_var2.max(), N_v_FD)
dEnu = safe_make_dx(all_Enu)

# Flux objects for oscillation analysis
phi_ND = nd_flux(pid.numu, all_Enu)
phi_FD = fd_flux(pid.numu, all_Enu.numpy()) # oscillation code doesn't support torch
assert in_lepton_region(samp_Emu, samp_cost, V1_ACCEPT, V2_ACCEPT).all()


In [ ]:
# Paper 1 clamps negative learned cross sections at FD inference only.
model_calc_fd = ModelCalc(model, enumax=10, unique=(N_SF == 3), force_ds_pos=True, device="cpu")
# total xsec integrand for toy model
I  = make_I(truth, all_Enu, all_v1, all_v2, flavor=pid.mu, kin_mapping=kin_mapping) # ~ Enu

# total xsec integrand for ML model
I_ = make_I(model_calc_fd, all_Enu, all_v1, all_v2, flavor=pid.mu, kin_mapping=kin_mapping) # ~ Enu

In [ ]:
# differential xsec for toy model
dsigma_samp  = make_dsigma_samp(truth, all_Enu, samp_v1, samp_v2, flavor=pid.mu, kin_mapping=kin_mapping) # ~ samp, Enu

# differential xsec for ML model
dsigma_samp_ = make_dsigma_samp(model_calc_fd, all_Enu, samp_v1, samp_v2, flavor=pid.mu, kin_mapping=kin_mapping) # ~ samp, Enu

In [ ]:
# plot total cross section

plt.plot(all_Enu, I/I.sum()/dEnu)
plt.plot(all_Enu, I_/I_.sum()/dEnu)
plt.xlabel(r'$E_\nu$ [GeV]')
plt.ylabel(r'$\sigma(E_\nu) / \int dE_\nu \sigma(E_\nu)$')
plt.ylim(0,0.25)
plt.show()

In [ ]:
# plot averaged differential xsec

mean_dsigma = dsigma_samp.mean(0).detach().numpy()
mean_dsigma_ = dsigma_samp_.mean(0).detach().numpy()
plt.plot(all_Enu, mean_dsigma/mean_dsigma.sum()/dEnu)
plt.plot(all_Enu, mean_dsigma_/mean_dsigma_.sum()/dEnu)
plt.xlabel(r'$E_\nu$ [GeV]')
plt.ylabel(r'$<d^2\sigma> / \int dE_\nu <d^2\sigma>$')
plt.show()

In [ ]:
# Report the raw model's pathologies before inference-side positivity.
I_raw = make_I(model_calc, all_Enu, all_v1, all_v2, flavor=pid.mu, kin_mapping=kin_mapping)
dsigma_raw = make_dsigma_samp(model_calc, all_Enu, samp_v1, samp_v2,
                              flavor=pid.mu, kin_mapping=kin_mapping)
raw_event_rate = dsigma_raw @ torch.as_tensor(phi_FD)
print("Raw negative cross-section entries:", float((dsigma_raw < 0).double().mean()))
print("Nonpositive raw integrated event densities:", int((raw_event_rate <= 0).sum()), "/", N_samps)
print("FD inference uses clamped cross sections and rejects invalid likelihoods.")


In [ ]:
# Oscillation model

s23_true = 0.748999
d31_true = 2.534e-3
dcp_true = 3.08923

def compute_osc_prob(enu, s23=0.748999, d31=2.534e-3, dcp=3.08923):
    baseline = 1300
    s12 = 0.554076
    s13 = 0.148155
    d21 = 7.49e-5
    # dCP = 177./180*np.pi = 3.08923

    prob = Probability_Matter_LBL(s12**2, s13**2, s23**2,
                                    dcp, d21, d31,
                                    baseline, enu)
    return prob

In [ ]:
# A 1,000-event run needs a wider search than the 6,200-event paper plot.
# The upper octant is retained, as in this original tutorial.
s23_2sq_true = 1 - (2 * s23_true**2 - 1)**2
stat_scale = np.sqrt(6200 / N_samps)
n_scan = max(81, 2 * cfg["osc"] + 1)
all_s23_2sq = torch.linspace(max(0.0, 1 - 0.07 * stat_scale), 1., n_scan)
all_d31 = torch.linspace(max(1e-4, d31_true - 0.00012 * stat_scale),
                         d31_true + 0.00012 * stat_scale, n_scan)

def rebuild_parameter_grid():
    x, y = torch.meshgrid(all_s23_2sq, all_d31, indexing='ij')
    return x, y, torch.sqrt(0.5 * (1 + torch.sqrt(1 - x)))

grid_s23_2sq, grid_d31, grid_s23 = rebuild_parameter_grid()
print("Search bounds:", (float(all_s23_2sq.min()), float(all_s23_2sq.max())),
      (float(all_d31.min()), float(all_d31.max())))


In [ ]:
# compute oscillated flux from unoscillated flux and osc params

def make_phi_FD(phi_ND_mu, phi_ND_e, all_Enu, s23, d31=d31_true, dcp=dcp_true, lepton="mu"):
    p_osc = compute_osc_prob(grab(all_Enu), s23=s23, d31=d31, dcp=dcp)
    assert not np.isclose(phi_ND_mu.sum() * safe_make_dx(all_Enu), 1) # DON'T PASS PDF
    assert not np.isclose(phi_ND_e.sum() * safe_make_dx(all_Enu), 1) # DON'T PASS PDF
    # nufast's torch port returns tensors regardless of input type, and
    # numpy >= 2 no longer broadcasts ndarray * Tensor — do the algebra in
    # torch and convert back to the caller's type at the end.
    was_torch = isinstance(phi_ND_mu, torch.Tensor)
    phi_mu_t = torch.as_tensor(phi_ND_mu)
    phi_e_t  = torch.as_tensor(phi_ND_e)

    if lepton=="mu":
        #phi_FD_mu = phi_ND_mu * p_osc[mu->mu] + phi_ND_e * p_osc[e->mu]
        phi_FD = phi_mu_t * p_osc[1][1] + phi_e_t * p_osc[0][1]
    if lepton=="e":
        #phi_FD_e  = phi_ND_mu * p_osc[mu->e]  + phi_ND_e * p_osc[e->e]
        phi_FD = phi_mu_t * p_osc[1][0]  + phi_e_t * p_osc[0][0]
    phi_FD = torch.nan_to_num(phi_FD, 0)
    return phi_FD if was_torch else grab(phi_FD)

In [ ]:
def compute_osc_grid(grid_s23, grid_d31, dsigma_samp, I, all_Enu,
                     phi_ND_mu, phi_ND_e=None, lepton="mu", bs_tensor=None):
    if lepton != "mu" or bs_tensor is not None:
        raise ValueError("This tutorial scans the single muon channel before bootstrapping.")
    return likelihood_grid(dsigma_samp, I, all_Enu, phi_ND_mu,
                           grid_s23, grid_d31, compute_osc_prob, phi_e=phi_ND_e)


In [ ]:
# compute log-likelihoods for true/ML xsec (p/q) and true/model oscillation parameters (_true/_model)

# log P({v_i}|p*)   : true ds, true flux
ll_p_true = shape_log_likelihood(dsigma_samp, I, phi_FD, dEnu)

# log P({v_i}|p_w)  : true ds, model flux (in loop)
ll_p_model = compute_osc_grid(grid_s23=grid_s23, grid_d31=grid_d31,
    dsigma_samp=dsigma_samp, I=I, phi_ND_mu=phi_ND, all_Enu=all_Enu) #, bs_tensor=bs_tensor)

# # log P({v_i}|q_w)  : model ds, model flux (in loop)
ll_q_model = compute_osc_grid(grid_s23=grid_s23, grid_d31=grid_d31,
    dsigma_samp=dsigma_samp_, I=I_, phi_ND_mu=phi_ND, all_Enu=all_Enu) #, bs_tensor=bs_tensor)

In [ ]:
# Profile likelihoods: minimize over the other parameter rather than averaging logs.
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for likelihood, label in ((ll_p_model, "true cross section"), (ll_q_model, "learned cross section")):
    nll = -likelihood.sum(0)
    nll = nll - nll.min()
    axes[0].plot(all_s23_2sq, 2 * nll.min(1).values, label=label)
    axes[1].plot(1e3 * all_d31, 2 * nll.min(0).values, label=label)
axes[0].set(xlabel=r'$\sin^2(2\theta_{23})$', ylabel=r'$2\Delta\mathrm{NLL}$')
axes[1].set(xlabel=r'$\Delta m^2_{31}$ [$10^{-3}$ eV$^2$]', ylabel=r'$2\Delta\mathrm{NLL}$')
axes[0].legend()
fig.tight_layout()
plt.show()


In [ ]:
# 2D LL distribution

# toy model
print("true xsec sample log-likelihood")
plt.imshow(ll_p_model.mean(0))
plt.show()

# ML model
print("learned xsec sample log-likelihood")
plt.imshow(ll_q_model.mean(0))
plt.show()

In [ ]:
# Standard bootstrap
print(len(trim), N_samps)
np.random.seed(12)
bs_tensor = torch.tensor(precompute_bootstrap_tensor(N_samps, N_boots))

In [ ]:
# Reused below for evaluating a KDE on a display grid independent of the scan.
make_kde = bootstrap_kde


In [ ]:
# Fit bootstraps in chunks, and expand numerical bounds if any optimum hits them.
# sin^2(2 theta23)=1 is a physical boundary and may legitimately accumulate fits.
for expansion in range(4):
    min_s23_p, min_d31_p = bootstrap_minima(ll_p_model, bs_tensor, grid_s23_2sq, grid_d31)
    min_s23_q, min_d31_q = bootstrap_minima(ll_q_model, bs_tensor, grid_s23_2sq, grid_d31)
    edges_p = boundary_counts(min_s23_p, min_d31_p, all_s23_2sq, all_d31)
    edges_q = boundary_counts(min_s23_q, min_d31_q, all_s23_2sq, all_d31)
    print("Boundary counts (truth / learned):", edges_p, edges_q)
    low_x = edges_p['x_lower'] + edges_q['x_lower'] > 0 and all_s23_2sq[0] > 0
    low_y = edges_p['y_lower'] + edges_q['y_lower'] > 0
    high_y = edges_p['y_upper'] + edges_q['y_upper'] > 0
    if not (low_x or low_y or high_y):
        break
    if expansion == 3:
        raise RuntimeError("Bootstrap fits still hit numerical scan bounds; widen the search.")
    dx = float(all_s23_2sq[-1] - all_s23_2sq[0])
    dy = float(all_d31[-1] - all_d31[0])
    all_s23_2sq = torch.linspace(max(0., float(all_s23_2sq[0]) - (0.5 * dx if low_x else 0.)), 1., n_scan)
    all_d31 = torch.linspace(max(1e-4, float(all_d31[0]) - (0.5 * dy if low_y else 0.)),
                             float(all_d31[-1]) + (0.5 * dy if high_y else 0.), n_scan)
    grid_s23_2sq, grid_d31, grid_s23 = rebuild_parameter_grid()
    ll_p_model = compute_osc_grid(grid_s23, grid_d31, dsigma_samp, I, all_Enu, phi_ND)
    ll_q_model = compute_osc_grid(grid_s23, grid_d31, dsigma_samp_, I_, all_Enu, phi_ND)

all_fits = np.concatenate([np.column_stack([min_s23_p, min_d31_p]),
                           np.column_stack([min_s23_q, min_d31_q])])
xlim, ylim = bootstrap_plot_limits(all_fits, np.array([s23_2sq_true, d31_true]))
plot_x = np.linspace(*xlim, 240)
plot_y = np.linspace(*ylim, 240)
plot_grid_x, plot_grid_y = np.meshgrid(plot_x, plot_y, indexing='ij')
kde_exact = make_kde(min_s23_p, min_d31_p, plot_x, plot_y)
kde_model = make_kde(min_s23_q, min_d31_q, plot_x, plot_y)

summary = {}
for name, x, y in (("true", min_s23_p, min_d31_p), ("learned", min_s23_q, min_d31_q)):
    summary[name] = dict(median=[float(np.median(x)), float(np.median(y))],
                         std=[float(np.std(x)), float(np.std(y))],
                         q16_q84=np.quantile(np.column_stack([x, y]), [0.16, 0.84], axis=0).tolist())
    print(name, summary[name])
np.savez(OUT / "bootstrap_fits.npz", true_x=min_s23_p, true_y=min_d31_p,
         learned_x=min_s23_q, learned_y=min_d31_q, injected=[s23_2sq_true, d31_true])
# Inputs permit FD convergence/plotting checks without another expensive training run.
np.savez(OUT / "fd_inputs.npz", enu=all_Enu, phi=phi_ND, samples=trim,
         dsigma_truth=dsigma_samp, dsigma_model=dsigma_samp_, dsigma_raw=dsigma_raw,
         I_truth=I, I_model=I_, I_raw=I_raw, bootstrap=bs_tensor)
report = dict(settings=cfg, summary=summary, bounds=dict(truth=edges_p, learned=edges_q),
              plot_limits=[xlim, ylim], raw_negative_entries=float((dsigma_raw < 0).double().mean()),
              raw_nonpositive_events=int((raw_event_rate <= 0).sum()),
              note="Current library toy model and VarPro training; not an exact paper-1 reproduction.")
(OUT / "analysis.json").write_text(json.dumps(report, indent=2) + "\n")


In [ ]:
# scatterplot of osc param minima

plt.scatter(min_s23_p, min_d31_p, alpha=.1)
plt.scatter(min_s23_q, min_d31_q, alpha=.1)
plt.xlabel(r'$\sin^2(2\theta_{23})$')
plt.ylabel(r'$\Delta m^2(\theta_{31})$')
plt.show()

## Oscillation parameters: a readable comparison

The display limits below follow **both** bootstrap clouds and the injected point. The likelihood scan is checked separately for clipped optima and expanded if needed. The mass-splitting data are multiplied by $10^3$ to match the axis units. The physical upper boundary $\sin^2(2\theta_{23})=1$ is retained.

The paper's Fig. 6 uses 6,200 FD events and 25,000 bootstrap draws. Quick runs with 1,000 events have roughly $\sqrt{6200/1000}\simeq2.49$ times larger statistical errors; increasing ND training steps cannot remove this. The current tutorial also uses a different toy-model prescription, training grid and trainer from the paper. See [the reproduction audit](docs/tutorial1-reproduction.md).


In [ ]:
sigma_cuts = [0.68, 0.95, 0.997]
def find_ci_levels(X, cuts=sigma_cuts, verbose=True):
    X_sorted = np.sort(X.ravel())[::-1]
    X_cdf = np.cumsum(X_sorted) / X.sum()
    assert np.isclose(X_cdf[-1], 1)
    levels = []
    for cut in cuts:
        idx = np.argmax(X_cdf >= cut)
        levels.append(X_sorted[idx])
    if verbose:
        for cut,level in zip(cuts,levels):
            print(cut, X[X>level].sum() / X.sum())
            # print(cut, (X>level).mean())
    return levels

def compute_line_intersection(m1, b1, m2, b2):
    x = (b2 - b1) / (m1 - m2)
    y = m1 * x + b1
    return (x, y)

def calculate_label_positions(cs, x0,y0,x1,y1):
    dx=x1-x0
    dy=y1-y0
    m = dy/dx
    b = y0 - m * x0

    label_positions = []

    # for vertices in CS.allsegs:
    for path in CS.get_paths():
        vertices = path.vertices
        for ii, ((x1,y1), (x2,y2)) in enumerate(zip(vertices[:-1], vertices[1:])):
            m_ = (y2-y1)/(x2-x1)
            b_ = y1 - m_*x1
            xi, yi = compute_line_intersection(m,b, m_,b_)
            # if min(x1,x2) < xi < max(x1,x2) and min(y1,y2) < yi < max(y1,y2):
            if ((x1<xi<x2) or (x2<xi<x1)) and ((y1<yi<y2) or (y2<yi<y1)):
                dx_, dy_ = xi-x0, yi-y0
                if (dx_*dx + dy_*dy) > 0:  # dot
                    label_positions.append((xi,yi))
                    break
    return label_positions

In [ ]:
from matplotlib.lines import Line2D
with plt.rc_context(rc=plot_rc_params):
    fig, axes = plt.subplots(2, 2, figsize=(6.5, 5.5), dpi=160,
        sharey='row', sharex='col',
        gridspec_kw=dict(hspace=0.04, wspace=0.04, width_ratios=[6, 1.3], height_ratios=[1.3, 6]))
    fig.delaxes(axes[0, 1])
    ax = axes[1, 0]
    for density, color, style in ((kde_exact, 'black', '--'), (kde_model, 'tab:blue', '-')):
        if density is not None:
            levels = sorted(set(find_ci_levels(density, verbose=False)))
            ax.contour(plot_grid_x, 1e3 * plot_grid_y, density,
                       levels=levels, colors=color, linestyles=style, linewidths=1.6)
    if kde_exact is None or kde_model is None:
        ax.scatter(min_s23_p, 1e3 * min_d31_p, s=8, alpha=.35, color='black')
        ax.scatter(min_s23_q, 1e3 * min_d31_q, s=8, alpha=.35, color='tab:blue')
    ax.plot(s23_2sq_true, 1e3 * d31_true, '*', color='black', markersize=11)
    ax.set(xlim=xlim, ylim=np.asarray(ylim) * 1e3,
           xlabel=r'$\sin^2(2\theta_{23})$',
           ylabel=r'$\Delta m^2_{31}$ [$10^{-3}$ eV$^2$]')
    ax.legend(handles=[Line2D([], [], color='black', linestyle='--', label='True cross section'),
                       Line2D([], [], color='tab:blue', label='Learned cross section'),
                       Line2D([], [], color='black', marker='*', linestyle='', label='Injected')],
              loc='lower left', fontsize=8)
    # Histograms span every sample, including estimates exactly at the physical boundary.
    xbins = np.linspace(*xlim, 26)
    ybins = np.linspace(*(np.asarray(ylim) * 1e3), 26)
    for x, y, color in ((min_s23_p, min_d31_p, 'black'), (min_s23_q, min_d31_q, 'tab:blue')):
        axes[0, 0].hist(x, bins=xbins, histtype='step', density=True, color=color)
        axes[1, 1].hist(1e3 * y, bins=ybins, histtype='step', density=True,
                        orientation='horizontal', color=color)
    axes[0, 0].axvline(s23_2sq_true, color='black', linewidth=1)
    axes[1, 1].axhline(1e3 * d31_true, color='black', linewidth=1)
    axes[0, 0].axis('off')
    axes[1, 1].axis('off')
    axes[0, 0].set_title(f"{N_samps:,} FD events | {N_boots:,} bootstrap draws", fontsize=11)
    fig.savefig(OUT / 'FD-inference-CIs.pdf', bbox_inches='tight')
    fig.savefig(OUT / 'FD-inference-CIs.png', bbox_inches='tight')
    plt.show()
print("Final comparison:", OUT / 'FD-inference-CIs.png')


The learned and truth fits need not coincide in a short run. Check the ND residuals, total-cross-section comparison, and grid resolution first. Tutorial 2 extends the workflow to both beam modes, electron appearance, charge mixing, and three oscillation parameters.